In [19]:
# ==============================================================================
# 📦 1. REQUIRED LIBRARIES IMPORT & COLAB GUARD
# ==============================================================================
import os
import datetime
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output
from tensorflow.keras.models import load_model

# 💡 TensorFlow ရဲ့ Warning စာသားတွေကို UI ထဲမရောက်အောင် ပိတ်ပစ်မည့် ကုဒ် (အပေါ်ဆုံးတွင်ထားပါ)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')

# 💡 HUGGING FACE CRASH PROTECTION: Colab Environment ဟုတ်မဟုတ် အရင်စစ်ဆေးခြင်း
is_colab = False
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

# ==============================================================================
# 🌐 2. DYNAMIC PATH ENVIRONMENT GUARD (AUTO-DETECT)
# ==============================================================================
if is_colab:
    print("🔗 Google Colab Environment တည်ရှိနေသဖြင့် Drive ကို ချိတ်ဆက်နေပါသည်...")
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    base_models_dir = '/content/drive/MyDrive/NewModel'
    meta_path = '/content/drive/MyDrive/NewModel/Station_Meta.csv'
    csv_path = '/content/drive/MyDrive/NewModel/Station_Data.csv'
else:
    print("🚀 Hugging Face (Voila Server) ပတ်ဝန်းကျင်ဖြစ်သဖြင့် Workspace ထဲမှ တိုက်ရိုက်ဖတ်ရှုနေပါသည်...")
    base_models_dir = '.'
    meta_path = 'Station_Meta.csv'
    csv_path = 'Station_Data.csv'

# စာလုံးအကြီးအသေး ကွဲလွဲမှုများအတွက် ကာကွယ်ခြင်း (Hugging Face Case-Sensitive အတွက်)
if not is_colab:
    if not os.path.exists(meta_path) and os.path.exists('station_Meta.csv'): meta_path = 'station_Meta.csv'
    if not os.path.exists(csv_path) and os.path.exists('station_Data.csv'): csv_path = 'station_Data.csv'

# ==============================================================================
# 💾 3. DATA LOADER & DUMMY FALLBACK (PRINT STATEMENTS HIDDEN FOR CLEAN UI)
# ==============================================================================
if os.path.exists(meta_path):
    meta_df = pd.read_csv(meta_path)
    meta_df.columns = [c.strip().capitalize() if c.strip().lower() == 'station' else c for c in meta_df.columns]
    # 💡 စာသားဖျောက်ရန် အောက်ပါ print ကို comment ပိတ်ထားပါသည်
    print(f"✅ Metadata ဖတ်ရှုမှု အောင်မြင်သည်။")
else:
    # ⚠️ ဒေတာအတုသုံးရပါက Notebook ထဲတွင်သာ သိစေရန် console တွင်ထုတ်ပါသည်
    import sys
    print("⚠️ Metadata ဖိုင်ကို ရှာမတွေ့ပါ။ Backup Dummy ဒေတာကို အသုံးပြုပါမည်။", file=sys.stderr)
    dummy_data = {
        'Station': ['Myitkyina', 'Mandalay', 'Sagaing', 'Myinmu', 'Pakokku', 'NyaungOo', 'Chauk', 'Minbu', 'Magway', 'Aunglan', 'Pyay', 'Seiktha', 'Hinthada', 'Zalun'],
        'Danger Level_cm': [1200, 1260, 1150, 1150, 2150, 2120, 1450, 1700, 1700, 2550, 2900, 1200, 1342, 1160],
        'Latitude': [25.3833, 21.9747, 21.8781, 21.9167, 21.3467, 21.1961, 20.8833, 20.1833, 20.1500, 19.3667, 18.8167, 18.1167, 17.6500, 17.4833],
        'Longitude': [97.4000, 96.0836, 95.9797, 95.5667, 95.0833, 94.9044, 94.8167, 94.8833, 94.9333, 95.2167, 95.2167, 95.1167, 95.4500, 95.5167]
    }
    meta_df = pd.DataFrame(dummy_data)

if os.path.exists(csv_path):
    ts_extended = pd.read_csv(csv_path)
    ts_extended['Date'] = pd.to_datetime(ts_extended['Date'])
    # 💡 စာသားဖျောက်ရန် အောက်ပါ print ကို comment ပိတ်ထားပါသည်
    print(f"✅ နေ့စဉ်ဒေတာဖိုင် (CSV) ဖတ်ရှုမှု အောင်မြင်သည်။")
else:
    import sys
    print("⚠️ ဒေတာဖိုင် ရှာမတွေ့ပါ။ စမ်းသပ်ရန် ဒေတာအတု ဖန်တီးပေးနေပါသည်။", file=sys.stderr)
    dates_idx = pd.date_range(end=datetime.date.today(), periods=35)
    ts_extended = pd.DataFrame({'Date': dates_idx})
    for st in meta_df['Station'].unique():
        ts_extended[f'{st}_WL'] = 400
        ts_extended[f'{st}_RF'] = 0.0

stations_list = meta_df['Station'].unique()

# ==============================================================================
# 🌧️ 4. OPEN-METEO WEATHER API CALL FUNCTION
# ==============================================================================
def get_weather_forecast_array(lat, lon, lt, base_date_str=None):
    if base_date_str:
        start_date = pd.to_datetime(base_date_str)
    else:
        start_date = datetime.datetime.now()

    days_to_get = int(lt)
    end_date = start_date + pd.Timedelta(days=days_to_get)

    start_str = start_date.strftime('%Y-%m-%d')
    end_str = end_date.strftime('%Y-%m-%d')

    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}&daily=precipitation_sum"
        f"&start_date={start_str}&end_date={end_str}&timezone=Asia/Yangon"
    )

    try:
        r = requests.get(url, timeout=10).json()
        if 'daily' in r and 'precipitation_sum' in r['daily']:
            p_list = r['daily']['precipitation_sum']
            cleaned_list = [float(x) if x is not None else 0.0 for x in p_list]

            if len(cleaned_list) >= days_to_get:
                return cleaned_list[:days_to_get]
            else:
                return cleaned_list + [0.0] * (days_to_get - len(cleaned_list))
    except Exception as e:
        print(f"⚠️ Weather API Error ({lat}, {lon}): {e}")

    return [0.0] * days_to_get

# ==============================================================================
# 📝 5. TAB 1: ENHANCED DATA ENTRY PANEL (MANUAL + FILE UPLOAD)
# ==============================================================================
style = {'description_width': 'initial'}
entry_date_picker = widgets.DatePicker(description='📅 Data Date:', value=datetime.date.today(), style=style)

file_uploader = widgets.FileUpload(
    accept='.csv, .xlsx, .xls',
    multiple=False,
    description='📁 Upload Excel/CSV',
    button_style='info',
    style=style
)

wl_inputs, rf_inputs = {}, {}
grid_rows = []

header_row = widgets.HBox([
    widgets.Label('📍 Station Name', layout=widgets.Layout(width='150px', font_weight='bold')),
    widgets.Label('🌊 WL (cm)', layout=widgets.Layout(width='120px', font_weight='bold')),
    widgets.Label('🌧️ RF (mm)', layout=widgets.Layout(width='120px', font_weight='bold'))
])
grid_rows.append(header_row)

for st in stations_list:
    wl_val = int(ts_extended.iloc[-1][f'{st}_WL']) if f'{st}_WL' in ts_extended.columns else 0
    rf_val = float(ts_extended.iloc[-1][f'{st}_RF']) if f'{st}_RF' in ts_extended.columns else 0.0

    wl_w = widgets.IntText(value=wl_val, layout=widgets.Layout(width='110px'))
    rf_w = widgets.FloatText(value=rf_val, layout=widgets.Layout(width='110px'))

    wl_inputs[st] = wl_w
    rf_inputs[st] = rf_w

    row = widgets.HBox([widgets.Label(st, layout=widgets.Layout(width='150px')), wl_w, rf_w])
    grid_rows.append(row)

update_btn = widgets.Button(description='💾 Update Manual Entry', button_style='primary', layout=widgets.Layout(margin='10px 0px 0px 10px'))
upload_btn = widgets.Button(description='🚀 Process Uploaded File', button_style='success', layout=widgets.Layout(margin='10px 0px 0px 10px'))
status_label = widgets.Label('')

def save_entry_to_system(b):
    global ts_extended
    target_date = pd.to_datetime(entry_date_picker.value)
    new_data = {'Date': target_date}
    for st in stations_list:
        new_data[f'{st}_WL'] = wl_inputs[st].value
        new_data[f'{st}_RF'] = rf_inputs[st].value
        for extra in ['Width', 'Elev']:
            if f'{st}_{extra}' in ts_extended.columns:
                new_data[f'{st}_{extra}'] = ts_extended.iloc[-1][f'{st}_{extra}']

    if target_date in ts_extended['Date'].values:
        idx = ts_extended[ts_extended['Date'] == target_date].index[0]
        for k, v in new_data.items(): ts_extended.at[idx, k] = v
        status_label.value = f"✅ {target_date.strftime('%Y-%m-%d')} ၏ ရှိပြီးသားဒေတာအား ပြင်ဆင်ပြီးပါပြီ။"
    else:
        ts_extended = pd.concat([ts_extended, pd.DataFrame([new_data])], ignore_index=True)
        status_label.value = f"✅ {target_date.strftime('%Y-%m-%d')} ဒေတာအသစ်အား စနစ်ထဲသို့ တိုးထည့်ပြီးပါပြီ။"

    try:
        ts_extended.to_csv(csv_path, index=False)
    except Exception as e:
        status_label.value = f"⚠️ Local Memory တွင်သာ မှတ်တမ်းတင်နိုင်ပါသည် - {e}"

def process_file_upload(b):
    global ts_extended
    if not file_uploader.value:
        status_label.value = "⚠️ ကျေးဇူးပြု၍ ဖိုင်အရင် ရွေးချယ်ပေးပါရန်။"
        return
    try:
        uploaded_file = list(file_uploader.value.values())[0]
        file_content = uploaded_file['content']
        file_name = uploaded_file['metadata']['name']
        import io
        if file_name.endswith('.csv'):
            uploaded_df = pd.read_csv(io.BytesIO(file_content))
        else:
            uploaded_df = pd.read_excel(io.BytesIO(file_content))

        if 'Date' not in uploaded_df.columns:
            status_label.value = "❌ ဖိုင်ထဲတွင် 'Date' Column မပါဝင်ပါ။"
            return

        uploaded_df['Date'] = pd.to_datetime(uploaded_df['Date'])
        ts_extended = ts_extended[~ts_extended['Date'].isin(uploaded_df['Date'])].copy()
        ts_extended = pd.concat([ts_extended, uploaded_df], ignore_index=True).sort_values('Date').reset_index(drop=True)
        ts_extended.to_csv(csv_path, index=False)
        status_label.value = f"✅ ဖိုင် '{file_name}' မှ ဒေတာများကို အောင်မြင်စွာ ပေါင်းထည့်ပြီးပါပြီ။"
    except Exception as e:
        status_label.value = f"❌ Error: {str(e)}"

update_btn.on_click(save_entry_to_system)
upload_btn.on_click(process_file_upload)

upload_box = widgets.VBox([widgets.Label('📋 Option B: Bulk Upload (Excel/CSV ဖိုင်တင်ရန်)'), widgets.HBox([file_uploader, upload_btn])])
manual_box = widgets.VBox([widgets.Label('📋 Option A: Manual Entry (လက်ရိုက်ထည့်ရန်)'), entry_date_picker, widgets.VBox(grid_rows), update_btn])
data_entry_panel = widgets.VBox([upload_box, widgets.HTML("<hr>"), manual_box, status_label])


# ==============================================================================
# 🔮 6. TAB 2: FORECAST SYSTEM ENGINE (TABLE ONLY)
# ==============================================================================
lt_forecast = widgets.Dropdown(options=[1, 3], value=3, description='🔮 Lead Time (Days):', style=style)
date_forecast = widgets.DatePicker(description='📅 Base Date:', value=datetime.date.today(), style=style)
forecast_btn = widgets.Button(description='🚀 Run Multi-Station Forecast', button_style='danger')
output_area_table = widgets.Output()

MODEL_CACHE = {}
SCALER_CACHE = {}
GLOBAL_GRAPHS_DATA = {}  # Tab 3 မှ လှမ်းသုံးရန် ဒေတာဗဟိုချက်

def run_forecast_table(b):
    global GLOBAL_GRAPHS_DATA, meta_df, ts_extended
    with output_area_table:
        clear_output()

        lt_val = int(lt_forecast.value)
        target_dt = pd.to_datetime(date_forecast.value)

        if os.path.exists(csv_path):
            ts_extended = pd.read_csv(csv_path)
            ts_extended['Date'] = pd.to_datetime(ts_extended['Date'])

        print(f"⏳ စခန်းအားလုံး၏ {lt_val}-Day Ahead Forecast ဇယားကို တွက်ချက်နေပါသည်...")

        summary_results = []
        GLOBAL_GRAPHS_DATA = {}

        for st_name in stations_list:
            try:
                st_info = meta_df[meta_df['Station'] == st_name].iloc[0]
                danger_level = st_info['Danger Level_cm']

                wl_col = f'{st_name}_WL'
                rf_col = f'{st_name}_RF'
                wd_col = f'{st_name}_Width'
                el_col = f'{st_name}_Elev'
                base_features = [wl_col, rf_col, wd_col, el_col]

                available_past_data = ts_extended[ts_extended['Date'] <= target_dt]
                current_window_df = available_past_data.tail(30).copy() if len(available_past_data) >= 30 else ts_extended.tail(30).copy()

                for col in base_features:
                    if col not in current_window_df.columns:
                        if '_Width' in col: current_window_df[col] = 100.0
                        elif '_Elev' in col: current_window_df[col] = 10.0
                        else: current_window_df[col] = 0.0

                forecasted_rainfalls = get_weather_forecast_array(st_info['Latitude'], st_info['Longitude'], lt_val, base_date_str=date_forecast.value)
                last_obs_wl = current_window_df.iloc[-1][wl_col]
                daily_change = last_obs_wl - current_window_df.iloc[-2][wl_col] if len(current_window_df) >= 2 else 0

                station_f_dates, f_wl, f_rf = [], [], []

                # Bias Correction Logic
                base_bias = 0.0
                yesterday_dt = target_dt - pd.Timedelta(days=1)
                past_actual_df = ts_extended[ts_extended['Date'] == target_dt]

                if not past_actual_df.empty:
                    actual_wl = past_actual_df.iloc[0][wl_col]
                    try:
                        m_1day_path = f"{base_models_dir}/{st_name}_model_1day.h5"
                        s_1day_path = f"{base_models_dir}/{st_name}_scaler_1day.pkl"
                        if os.path.exists(m_1day_path) and os.path.exists(s_1day_path):
                            if m_1day_path not in MODEL_CACHE:
                                from tensorflow.keras.layers import Dense
                                MODEL_CACHE[m_1day_path] = load_model(m_1day_path, compile=False, custom_objects={'Dense': lambda **kwargs: Dense(**{k: v for k, v in kwargs.items() if k != 'quantization_config'})})
                            if s_1day_path not in SCALER_CACHE:
                                SCALER_CACHE[s_1day_path] = joblib.load(s_1day_path)

                            m_1d, s_1d = MODEL_CACHE[m_1day_path], SCALER_CACHE[s_1day_path]
                            yesterday_window = ts_extended[ts_extended['Date'] <= yesterday_dt].tail(30)
                            if len(yesterday_window) >= 1:
                                n_feats_1d = s_1d.n_features_in_ if hasattr(s_1d, 'n_features_in_') else len(s_1d.scale_)
                                if n_feats_1d == 1:
                                    final_in_1d = np.hstack([s_1d.transform(yesterday_window[[wl_col]].values), yesterday_window[[rf_col, wd_col, el_col]].values])
                                    past_pred_wl = int(round(s_1d.inverse_transform(m_1d(np.expand_dims(final_in_1d, axis=0), training=False).numpy())[0, 0]))
                                else:
                                    past_pred_wl = int(round(m_1d(np.expand_dims(s_1d.transform(yesterday_window[base_features].values), axis=0), training=False).numpy()[0, 0] * s_1d.scale_[0] + s_1d.min_[0]))
                                base_bias = max(min(past_pred_wl - actual_wl, 50), -50)
                    except:
                        base_bias = 0.0

                for i in range(1, lt_val + 1):
                    curr_f_date = target_dt + pd.Timedelta(days=i)
                    m_path = f"{base_models_dir}/{st_name}_model_{i}day.h5"
                    if not os.path.exists(m_path): m_path = f"{base_models_dir}/{st_name}_model_1day.h5"
                    s_path = f"{base_models_dir}/{st_name}_scaler_1day.pkl"

                    if os.path.exists(m_path) and os.path.exists(s_path):
                        if m_path not in MODEL_CACHE:
                            from tensorflow.keras.layers import Dense
                            MODEL_CACHE[m_path] = load_model(m_path, compile=False, custom_objects={'Dense': lambda **kwargs: Dense(**{k: v for k, v in kwargs.items() if k != 'quantization_config'})})
                        if s_path not in SCALER_CACHE: SCALER_CACHE[s_path] = joblib.load(s_path)
                        m, s = MODEL_CACHE[m_path], SCALER_CACHE[s_path]

                        n_features = s.n_features_in_ if hasattr(s, 'n_features_in_') else len(s.scale_)
                        if n_features == 1:
                            final_input = np.hstack([s.transform(current_window_df[[wl_col]].values), current_window_df[[rf_col, wd_col, el_col]].values])
                            pred_wl_raw = int(round(s.inverse_transform(m(np.expand_dims(final_input, axis=0), training=False).numpy())[0, 0]))
                        else:
                            pred_wl_raw = int(round(m(np.expand_dims(s.transform(current_window_df[base_features].values), axis=0), training=False).numpy()[0, 0] * s.scale_[0] + s.min_[0]))

                        pred_wl = int(round(pred_wl_raw - (base_bias * (1.0 / i))))
                        if pred_wl <= 0 or abs(pred_wl - last_obs_wl) > 500:
                            pred_wl = int(last_obs_wl + (daily_change * i * 0.3))
                    else:
                        pred_wl = int(last_obs_wl + (daily_change * i * 0.3))

                    station_f_dates.append(curr_f_date)
                    f_wl.append(pred_wl)
                    f_rf.append(forecasted_rainfalls[i - 1])

                    new_row = current_window_df.iloc[-1].copy()
                    new_row['Date'] = curr_f_date
                    new_row[wl_col] = pred_wl
                    current_window_df = pd.concat([current_window_df.iloc[1:], pd.DataFrame([new_row])])

                final_pred_wl = f_wl[-1]
                diff_from_danger = final_pred_wl - danger_level
                status = "🚨 ရေကြီးနိုင်သည်" if final_pred_wl >= danger_level else "🟢 စိတ်ချရ"
                wl_net_change = final_pred_wl - last_obs_wl
                wl_change_display = f"📈 တက်မည် (+{int(wl_net_change)} cm)" if wl_net_change > 0 else (f"📉 ကျမည် ({int(wl_net_change)} cm)" if wl_net_change < 0 else "⚖️ မပြောင်းလဲပါ")

                summary_results.append({
                    "📍 Station": st_name, "📉 Last Observed WL (cm)": last_obs_wl, "⚠️ 24-hr Change (cm)": f"+{int(daily_change)}" if daily_change > 0 else f"{int(daily_change)}",
                    "🌧️ Forecast RF (mm)": round(sum(f_rf), 1), f"🔮 Forecasted WL ({lt_val}-Day)": final_pred_wl, "📈/📉 Forecast WL Change": wl_change_display,
                    "⚠️ Danger Level (cm)": danger_level, "➕/➖ Diff (cm)": f"+{diff_from_danger}" if diff_from_danger >= 0 else f"{diff_from_danger}", "📢 Status": status
                })

                GLOBAL_GRAPHS_DATA[st_name] = {
                    'f_dates': station_f_dates, 'f_wl': f_wl, 'f_rf': f_rf, 'danger_level': danger_level,
                    'recent_df': ts_extended[ts_extended['Date'] <= target_dt].tail(15).copy(), 'last_obs_wl': last_obs_wl
                }
            except Exception as e:
                print(f"❌ Error in {st_name}: {e}")

        if summary_results:
            print(f"\n📊 Summary Forecast Table for All Stations")
            display(pd.DataFrame(summary_results))
            print("\n💡 Graph များကြည့်ရှုရန် '📈 Graph များ သီးသန့်ကြည့်ရှုရန်' Tab ထဲသို့ သွားရောက်ကြည့်ရှုနိုင်ပါသည်ဗျာ။")

forecast_btn.on_click(run_forecast_table)
forecast_panel = widgets.VBox([widgets.HBox([date_forecast, lt_forecast]), forecast_btn, output_area_table])


# ==============================================================================
# 📈 7. TAB 3: NEW SELECTIVE GRAPH VIEW PANEL (သီးသန့် GRAPH ကြည့်ရန်စာမျက်နှာ)
# ==============================================================================
graph_station_selector = widgets.Dropdown(
    options=['All Stations (အားလုံးပြရန်)'] + list(stations_list),
    value='All Stations (အားလုံးပြရန်)',
    description='📍 Select Station:',
    style=style
)
show_graph_btn = widgets.Button(description='📊 Show Graph', button_style='success')
output_area_graph = widgets.Output()

def render_graphs_on_demand(b):
    with output_area_graph:
        clear_output()
        if not GLOBAL_GRAPHS_DATA:
            print("⚠️ ကျေးဇူးပြု၍ '🔮 ခန့်မှန်းချက်တွက်ထုတ်ရန်' Tab တွင် Run Forecast ကို အရင်နှိပ်ပေးပါရန်။")
            return

        selected_st = graph_station_selector.value
        stations_to_plot = stations_list if selected_st == 'All Stations (အားလုံးပြရန်)' else [selected_st]

        print(f"📈 {selected_st} ၏ ရေမှတ်ပြဂရပ်များကို ဆွဲနေပါသည်...")
        for st_name in stations_to_plot:
            if st_name in GLOBAL_GRAPHS_DATA:
                p_pkg = GLOBAL_GRAPHS_DATA[st_name]
                r_df = p_pkg['recent_df']
                all_dates_str = r_df['Date'].dt.strftime('%Y-%m-%d').tolist() + [d.strftime('%Y-%m-%d') for d in p_pkg['f_dates']]
                x_idx = np.arange(len(all_dates_str))

                fig, ax1 = plt.subplots(figsize=(12, 3))
                ax1.plot(x_idx[:len(r_df)], r_df[f'{st_name}_WL'].values, 'o-', label='Past Observed', color='#1f77b4', alpha=0.5)
                ax1.plot(x_idx[len(r_df)-1:], [p_pkg['last_obs_wl']] + p_pkg['f_wl'], 'rs--', label='AI Forecast', linewidth=2)
                ax1.axhline(y=p_pkg['danger_level'], color='red', linestyle='-', label=f'Danger Level ({p_pkg["danger_level"]})')
                ax1.set_ylabel("Water Level (cm)")

                ax2 = ax1.twinx()
                ax2.bar(x_idx, [0]*len(r_df) + p_pkg['f_rf'], color='blue', alpha=0.2, label='Forecast RF')
                ax2.set_ylim(max(max(p_pkg['f_rf'])*5, 100), 0)
                ax2.set_ylabel("Rainfall (mm)", color='blue')

                ax1.set_xticks(x_idx); ax1.set_xticklabels(all_dates_str, rotation=90, fontsize=8)
                plt.title(f"📍 Station: {st_name}")
                plt.show()
                print("-" * 100)

show_graph_btn.on_click(render_graphs_on_demand)
graph_panel = widgets.VBox([
    widgets.HTML("<h4>📈 စခန်းအလိုက် Forecast Graph များကို သီးသန့်ရွေးချယ်ကြည့်ရှုခြင်း</h4>"),
    widgets.HBox([graph_station_selector, show_graph_btn]),
    output_area_graph
])


# ==============================================================================
# 🎨 PREMIUM CSS INTERFACE STYLING
# ==============================================================================
custom_tab_css = widgets.HTML('''
<style>
    .jupyter-widgets.widget-tab > .p-TabBar {
        background-color: #f8f9fa !important;
        border-bottom: 2px solid #0b6623 !important;
        display: flex !important;
        flex-wrap: nowrap !important;
        min-height: 48px !important;
    }
    .jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-item,
    .jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-tab {
        min-width: max-content !important;
        max-width: none !important;
        width: auto !important;
        flex: 0 0 auto !important;
        height: 42px !important;
        line-height: 20px !important;
        padding: 10px 20px !important;
        margin-right: 6px !important;
        border-top-left-radius: 8px !important;
        border-top-right-radius: 8px !important;
        background-color: #e8eaed !important;
        box-sizing: border-box !important;
    }
    .p-TabBar-itemLabel, .p-TabBar-tabLabel {
        overflow: visible !important;
        white-space: nowrap !important;
        display: inline-block !important;
        font-family: 'Pyidaungsu', 'Segoe UI', sans-serif !important;
        font-size: 13px !important;
        font-weight: 500 !important;
        color: #333333 !important;
        vertical-align: middle !important;
    }
    .jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-item:hover,
    .jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-tab:hover {
        background-color: #e2e4e7 !important;
        color: #0b6623 !important;
        cursor: pointer;
    }
    .jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-item.p-mod-current,
    .jupyter-widgets.widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current {
        background-color: #ffffff !important;
        border: 1px solid #0b6623 !important;
        border-bottom: none !important;
        border-top: 4px solid #0b6623 !important;
        color: #0b6623 !important;
    }
</style>
''')
display(custom_tab_css)

# ==============================================================================
# 🎨 DMH LOGO + FIXED MYANMAR FONT TITLE SYSTEM (CHROME & EDGE COMPATIBLE)
# ==============================================================================
import base64

# မိမိ Drive ထဲက Logo ဖိုင်အမည် (ဥပမာ - logo.jpg သို့မဟုတ် dmh_logo.png)
logo_file_name = 'ldkGxgvcn7BhYmAtS'
local_logo_path = f'/content/drive/MyDrive/NewModel/DMH_Logo.png'
logo_src = "https://share.google/ldkGxgvcn7BhYmAtS"

if os.path.exists(local_logo_path):
    with open(local_logo_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode()
        ext = logo_file_name.split('.')[-1].lower()
        mime_type = "image/png" if ext == 'png' else "image/jpeg"
        logo_src = f"data:{mime_type};base64,{encoded_string}"

# 🌟 စာလုံးပေါင်းစုံစနစ် ကောင်းမွန်အောင် ပြင်ဆင်ထားသော HTML Title
app_title = widgets.HTML(
    value=f"""
    <div style="display: flex; align-items: center; justify-content: center; margin-bottom: 22px; gap: 18px; padding: 5px;">
        <img src="{logo_src}"
             alt="DMH Logo"
             style="height: 85px; width: auto; object-fit: contain; filter: drop-shadow(1px 1px 3px rgba(0,0,0,0.15));">

        <div style="text-align: left;">
            <div style="margin: 0 0 4px 0; color: #0b6623; font-family: 'Myanmar Sanam', 'Pyidaungsu', 'Segoe UI', 'Arial', sans-serif; font-size: 23px; font-weight: bold; line-height: 1.4; -webkit-font-smoothing: antialiased;">
                မိုးလေဝသနှင့်ဇလဗေဒညွှန်ကြားမှုဦးစီးဌာန
            </div>
            <h2 style="margin: 0; color: #1e5494; font-family: 'Segoe UI', 'Arial', sans-serif; font-size: 25px; font-weight: bold; line-height: 1.2;">
                DMH AI Flood Dashboard (Ayeyarwady River)
            </h2>
            <p style="margin: 3px 0 0 0; font-size: 20px; color: #666666; font-family: 'Pyidaungsu', 'Segoe UI', sans-serif; font-weight: normal;">
                [ ဧရာဝတီမြစ်၏ မြစ်ရေခန့်မှန်းရေးစာမျက်နှာ ]
            </p>
        </div>
    </div>
    """
)

# ==============================================================================
# 🚀 8. MAIN INTERACTIVE APP DISPLAY (BUILD TAB UI)
# ==============================================================================
final_app_ui = widgets.Tab(children=[data_entry_panel, forecast_panel, graph_panel])
final_app_ui.set_title(0, '📝 နေ့စဉ်မှတ်တမ်းသွင်းရန်')
final_app_ui.set_title(1, '🔮 ခန့်မှန်းချက်တွက်ရန်')
final_app_ui.set_title(2, '📈 Graph များကြည့်ရန်')
final_app_ui.layout.width = '100%'

main_interface = widgets.VBox([app_title, final_app_ui])
display(main_interface)


🔗 Google Colab Environment တည်ရှိနေသဖြင့် Drive ကို ချိတ်ဆက်နေပါသည်...
✅ Metadata ဖတ်ရှုမှု အောင်မြင်သည်။
✅ နေ့စဉ်ဒေတာဖိုင် (CSV) ဖတ်ရှုမှု အောင်မြင်သည်။


HTML(value="\n<style>\n    .jupyter-widgets.widget-tab > .p-TabBar {\n        background-color: #f8f9fa !impor…